# ADK 2.x Parallel Workflow Example

This notebook demonstrates how to build and execute a parallel processing workflow using the `google.adk` library. We will define multiple independent nodes that process data concurrently, use a `JoinNode` to synchronize them, and aggregate their results into a final output node.

In [1]:
import google.adk

print(google.adk.__version__)

2.5.0


In [2]:
from google.adk import Agent
from google.adk import Workflow
from google.adk import Event
from pydantic import BaseModel
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# Define the default model for potential agent nodes

In [3]:
MODEL = "gemini-2.5-flash"

## 1. Define Parallel Processing Nodes (A, B, C)

Here we define three independent functions (`node_A`, `node_B`, and `node_C`). In a real-world scenario, these nodes might execute complex independent tasks, make API calls, or run separate LLM prompts. 

Each node takes an input, performs a mathematical operation, and returns an `Event` object containing a status message and the computed output.

In [4]:
def node_A(node_input: str):
    return Event(
        message=f"Node A executed... input={node_input}", 
        output=int(node_input)
    )

def node_B(node_input: str):
    input_b = int(node_input)
    return Event(
        message=f"Node B executed... input={node_input}", 
        output=input_b * 100
    )

def node_C(node_input: str):
    input_c = int(node_input)
    return Event(
        message=f"Node C executed... input={node_input}", 
        output=input_c * input_c
    )

## 2. Define Aggregation and Display Node (D)

This node serves as our final step. Because it will be placed *after* a `JoinNode`, it expects its `node_input` to be a dictionary containing the outputs of all the previous parallel nodes. 

It safely extracts these values, calculates the total sum, and formats a markdown response to cleanly display the final execution results.

In [5]:
# ==========================================
# 2. DEFINE AGGREGATION & DISPLAY NODE (D)
# ==========================================
def node_D(node_input: dict) -> Event:
    """
    Collects outputs from JoinNode, calculates the sum, and displays it.
    """
    # Safely extract outputs with a fallback of 0.0
    val_a = node_input.get("node_A", 0.0)
    val_b = node_input.get("node_B", 0.0)
    val_c = node_input.get("node_C", 0.0)

    # Calculate the sum
    total_sum = val_a + val_b + val_c

    # Format a clean markdown message to show to the user
    display_message = (
        f"### Execution Complete!\n\n"
        f"Successfully collected parallel outputs:\n"
        f"- **Node A Output:** `{val_a}`\n"
        f"- **Node B Output:** `{val_b}`\n"
        f"- **Node C Output:** `{val_c}`\n\n"
        f"--- \n"
        f"### Result\n"
        f"**Node D (Total Sum):** `{total_sum}`"
    )

    # Returning an Event with a 'message' displays it to the user in the UI
    return Event(message=display_message, output=total_sum)

## 3. Construct the Workflow Routing

Finally, we assemble the individual nodes into a comprehensive `Workflow`. 
We create a `JoinNode` to synchronize the branches. The `edges` list defines the topology of our execution graph:
1. `START` triggers nodes A, B, and C in parallel.
2. The outputs of A, B, and C are routed into the `join_node`.
3. Once all parallel branches finish, the `join_node` passes the aggregated dictionary to `node_D`.

In [6]:
from google.adk.workflow import JoinNode

# Initialize the join node to synchronize parallel branches
join_node = JoinNode(name="join_node")

# Define the workflow topology
root_agent = Workflow(
    name="routing_workflow",
    edges=[
        ("START", node_A, join_node),
        ("START", node_B, join_node),
        ("START", node_C, join_node),
        (join_node, node_D),
    ],
)

## 4. Run the Workflow

To actually run this workflow, you can trigger the `root_agent` with an initial input string that all the parallel start nodes will receive.

In [7]:
# 1. Initialize an in-memory session service to track execution state
session_service = InMemorySessionService()

In [8]:
app_name = "parallel_workflow_app"
user_id = "local_user"
session_id = "session_01"

# 2. Await the session creation
session = await session_service.create_session(
    app_name=app_name, 
    user_id=user_id,
    session_id=session_id
)

In [9]:
# 3. Create a Runner and attach it to your workflow
runner = Runner(
    agent=root_agent,
    app_name=app_name,
    session_service=session_service
)

In [10]:
# Example execution input
initial_input_value = "5"

In [11]:
# 4. Format the input message using genai Content types
input_message = Content(role="user", parts=[Part(text=initial_input_value)])

print("Starting workflow execution...\n")

# 5. Execute the workflow asynchronously 
# Using async for ensures we properly await each yielded event from the runner
async for event in runner.run_async(
    user_id=session.user_id,
    session_id=session.id,
    new_message=input_message
):
    if event.message:
        print("-"*40)
        for text_part in event.message.parts:
            print(text_part.text)
        print("-"*40)

Starting workflow execution...

----------------------------------------
Node A executed... input=5
----------------------------------------
----------------------------------------
Node B executed... input=5
----------------------------------------
----------------------------------------
Node C executed... input=5
----------------------------------------
----------------------------------------
### Execution Complete!

Successfully collected parallel outputs:
- **Node A Output:** `5`
- **Node B Output:** `500`
- **Node C Output:** `25`

--- 
### Result
**Node D (Total Sum):** `530`
----------------------------------------
